# Burgers: error versus index set and viscosity

We learn the solution operator $K: u_0 \mapsto u(\cdot, T)$ of the viscous Burgers' equation
$u_t + u u_x = \nu u_{xx}$ on $(0,1)$, $u(0,t) = u(1,t) = 0$, with $T = 0.2$.

* Inputs and outputs are sine coefficients in the basis $\sqrt 2 \sin(n\pi x)$; data come from the spectral Galerkin solver in `burgers_solver.py`.
* Input measure: $d_{\mathrm{in}} = 20$ independent coefficients, the $n$-th distributed as $\mathrm{Jacobi}(\alpha_n, \alpha_n)$. **As run:** $\alpha_n = (n-1)^2$, $n = 1, \dots, 20$ (so the first mode is uniform on $[-1,1]$).
* Approximation space: polynomial operators $\psi_m\, p_\lambda(u_0)$ with $\lambda$ in an anisotropic index set with weights $w_j$ = `linspace(1, 0.01, 20)` (the weights enter as an exponent $1/w_j$, see `owls.index_set`) and univariate degree at most 10.

We compare weighted hyperbolic cross sets ("HC") with weighted $\ell^1$ sets ("lp" with $p=1$, i.e. $\sum_j \lambda_j^{1/w_j} \le k$) of similar size, for $\nu \in \{0.1, 0.01, 0.001\}$. Each operator is fit from $M = \lfloor N\log N\rfloor$ optimally sampled inputs and $d_{\mathrm{out}} = 150$ output modes, and tested on 1000 inputs $u_0 \sim \rho$ (relative empirical Bochner error).

**Status:** the data in `data/` are from the arXiv v1 run; this experiment was not rerun after the cleanup. Rerunning takes a long time (the largest index sets need about 58,000 PDE solves per viscosity).

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from burgers_solver import burgers_map, sine_basis_vandermonde

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
NUS = [0.1, 0.01, 0.001]
HC_PARAMS = [3.5, 11, 29, 70]   # hyperbolic cross parameters
LP_PARAMS = [1, 2, 3, 4]        # weighted l1 parameters
D_OUT = 150                     # output modes
D_SOLVE = 200                   # as run: the solver uses 10 * D_IN = 200 modes
T, DT = 0.2, 1e-4
MAX_SIZE = 10000
M_TEST = 1000
SEED = 0
if QUICK:
    NUS, HC_PARAMS, LP_PARAMS, M_TEST, T = [0.1], [3.5], [1], 50, 0.01

D_IN = 20                                             # input sine modes
MAX_DEGREE = 10                                       # max univariate polynomial degree
measures = [owls.Jacobi(a**2) for a in range(D_IN)]  # as run: alpha_n = (n - 1)^2 for n = 1..20
weights = np.linspace(1, 0.01, D_IN)                 # anisotropy weights

In [ ]:
def rel_error(nu, kind, k, rng):
    """Fit on M = N log N optimal samples; return the relative test error and N_eff."""
    L = owls.index_set(D_IN, k, kind, weights, MAX_DEGREE, MAX_SIZE, p=1)
    N = len(L)
    solver = lambda u0: burgers_map(u0, nu, T, DT, D_SOLVE, D_OUT)
    model = owls.learn_operator(solver, measures, L, M=int(N * np.log(N)), optimal=True, rng=rng)
    x_test = owls.sample_rho(M_TEST, measures, rng)
    return owls.relative_bochner_error(model.predict(x_test), solver(x_test)), N

if RECOMPUTE or QUICK:
    errs = {"hc": np.zeros((len(NUS), len(HC_PARAMS))), "lp": np.zeros((len(NUS), len(LP_PARAMS)))}
    dims = {"hc": np.zeros(len(HC_PARAMS)), "lp": np.zeros(len(LP_PARAMS))}
    for i, nu in enumerate(NUS):
        for kind, params in [("hc", HC_PARAMS), ("lp", LP_PARAMS)]:
            for j, k in enumerate(params):
                rng = np.random.default_rng([SEED, i, j, kind == "hc"])
                errs[kind][i, j], dims[kind][j] = rel_error(nu, kind, k, rng)
                print(f"nu = {nu}, {kind}({k}): N_eff = {dims[kind][j]:.0f}, relative error {errs[kind][i, j]:.3e}")
    for kind in ["hc", "lp"]:
        np.save(os.path.join(DATA, f"{kind}_errs.npy"), errs[kind])
        np.save(os.path.join(DATA, f"{kind}_dims.npy"), dims[kind])

In [ ]:
dims_hc, err_hc = np.load(os.path.join(DATA, "hc_dims.npy")), np.load(os.path.join(DATA, "hc_errs.npy"))
dims_lp, err_lp = np.load(os.path.join(DATA, "lp_dims.npy")), np.load(os.path.join(DATA, "lp_errs.npy"))
nus = [0.1, 0.01, 0.001][:err_hc.shape[0]]

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
plt.figure(figsize=(fig_width, fig_height), layout="constrained")
ax = plt.gca()
ax.set(xscale="log", yscale="log", xlabel=r"$N_{\mathrm{eff}}(k;\ast)$", ylabel=r"Relative $L^2_\rho(\mathcal X; \mathcal Y)$ Error")
styles = {0.001: ("-", "coral"), 0.01: ("--", "red"), 0.1: (":", "firebrick")}
for i in reversed(range(len(nus))):
    ls, color = styles[nus[i]]
    ax.plot(dims_lp, err_lp[i], "D" + ls, color=color, label=r"$\ast = 1, \nu = %s$" % nus[i], linewidth=1, markersize=3)
    ax.plot(dims_hc, err_hc[i], "X" + ls, color=color, label=r"$\ast = \mathrm{HC}, \nu = %s$" % nus[i], linewidth=1, markersize=3)
plt.legend(loc="best")
savefig(os.path.join(FIGS, "vB_Err.pdf"))
plt.show()